# EOP Lab 1: Evidence Chain Extraction

**Series**: Agentic AI Science Playbook -- EOP Domain
**Goal**: Given a messy research repository description, identify ECF's seven artifact types and recommend restructuring.
**Prerequisites**: Foundation Labs 0-4.
**Time**: ~60 min.

---

## Background: Evidence-Oriented Programming (EOP)

**EOP** reorients research software development around the relationship between *scientific claims* and the *computational artifacts* that substantiate them. The **Evidence Chain Formalization (ECF)** maps:

```
input data
  --> experimental/analytical process
    --> output data
      --> visual data
        --> plotting/summarizing process
          --> visual claims (figures, tables, statistics)
            --> documentation
```

**Seven ECF artifact types**:

| # | Artifact | Role |
|---|---------|------|
| 1 | Input data | Raw measurements or datasets |
| 2 | Experimental process | Code that runs the analysis |
| 3 | Output data | Computed intermediate results |
| 4 | Visual data | Data prepared for plotting |
| 5 | Plotting process | Code that produces figures |
| 6 | Visual claims | Figures, tables, statistics in the paper |
| 7 | Documentation | README, entry document, pipeline instructions |

---

## What You Will Build

An EOP agent with three tools:
- `identify_artifacts` -- classify files in a repo into ECF artifact types
- `suggest_structure` -- recommend ECF directory structure
- `validate_chain` -- check whether the evidence chain is complete

## Learning Objectives

By the end of this lab, you will be able to:
- Identify and classify all 7 ECF artifact types in a research repository
- Build an agent that automates evidence chain analysis
- Validate whether a scientific claim has complete computational support
- Recommend ECF-compliant directory structures for research projects

## Why This Matters

The reproducibility crisis affects over 70% of published computational research. Much of this stems from incomplete disclosure — researchers share code but not the data it processes, or share figures but not the scripts that generated them.

**EOP with ECF** provides a systematic framework for ensuring that every scientific claim is traceable back through the full computational chain. Your agent automates this analysis, turning what would be hours of manual auditing into seconds of automated classification.

> **Impact**: Journals like Nature and Science increasingly require reproducibility statements. An EOP agent can automatically verify that a submission meets disclosure requirements before review.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Foundation Labs 0-4 |
| Domain knowledge | None required — ECF framework is explained here |
| Reference | Zhang et al., "Research software as scientific evidence" |

**NVIDIA Connection**: This lab demonstrates how LLM agents can enforce **research quality standards** — a use case increasingly important as AI-generated research grows. The same pattern works with **NVIDIA NIM** endpoints, enabling institutions to deploy EOP compliance checking at scale.

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## Concept: From Files to Evidence

When you look at a research repository, you see files: `.py`, `.csv`, `.png`, `.ipynb`. But from an EOP perspective, each file is an **evidence artifact** with a specific role in the chain from data to claims.

The key insight is that **the same file type can serve different roles**:

| File | Could Be... | ECF Classification |
|------|-------------|-------------------|
| `data.csv` | Raw measurements | input_data |
| `data.csv` | Computed results | output_data |
| `data.csv` | Values for a figure | visual_data |
| `plot.py` | Generates Figure 1 | plotting_process |
| `plot.py` | Runs the experiment | experimental_process |

The agent must use **context** (file location, naming, surrounding files) to correctly classify each artifact — this is why LLM-powered classification is more effective than rule-based approaches.

### Define EOP Tool Schemas
Three tools for evidence chain analysis -- each with typed Pydantic arguments. The LLM will read these schemas to understand what each tool does and what arguments it needs. We define `IdentifyArtifactsArgs`, `SuggestStructureArgs`, and `ValidateChainArgs` along with the OpenAI-compatible tool definitions.

## Tool Schemas

In [ ]:
ArtifactType = Literal[
    "input_data", "experimental_process", "output_data",
    "visual_data", "plotting_process", "visual_claim", "documentation", "unknown"
]

class IdentifyArtifactsArgs(BaseModel):
    file_descriptions: list[str] = Field(
        ..., description="List of file names or brief descriptions to classify.")
    context: str | None = Field(None, description="Additional context about the research project.")

class SuggestStructureArgs(BaseModel):
    project_summary: str = Field(..., description="Brief description of the research project and its claims.")
    current_files: list[str] = Field(..., description="Current file/directory list.")

class ValidateChainArgs(BaseModel):
    artifacts_found: list[str] = Field(
        ..., description="List of ECF artifact types found in the repository.")
    claim_text: str = Field(..., description="The scientific claim being supported.")

OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "identify_artifacts",
        "description": "Classify repository files into ECF artifact types. Use when analyzing a research repository.",
        "parameters": IdentifyArtifactsArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "suggest_structure",
        "description": "Recommend an ECF-compliant directory structure for the project.",
        "parameters": SuggestStructureArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "validate_chain",
        "description": "Check whether all ECF artifact types are present for a given claim.",
        "parameters": ValidateChainArgs.model_json_schema()}},
]
SCHEMA_MAP = {
    "identify_artifacts": IdentifyArtifactsArgs,
    "suggest_structure": SuggestStructureArgs,
    "validate_chain": ValidateChainArgs,
}
print("EOP tools registered:", [t["function"]["name"] for t in OPENAI_TOOLS])

## EOP Agent

### Build the EOP Agent
This agent uses our three tools to analyze research repositories. It can answer both tool-calling questions and conceptual questions about EOP. The agent sends the user message to the LLM with our tool definitions, validates the returned arguments with Pydantic, and returns the parsed result.

In [ ]:
EOP_SYSTEM = """You are an Evidence-Oriented Programming (EOP) assistant. You help researchers:
1. Identify ECF artifacts in their research repositories
2. Suggest ECF-compliant directory structures
3. Validate that evidence chains are complete

ECF artifact types: input_data, experimental_process, output_data,
visual_data, plotting_process, visual_claim, documentation.

Use the provided tools for analysis tasks. For conceptual questions, answer directly."""

def eop_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": EOP_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "args": None, "content": msg.content}
    call = msg.tool_calls[0]
    raw = json.loads(call.function.arguments)
    validated = SCHEMA_MAP[call.function.name](**raw)
    return {"tool": call.function.name, "args": validated}

## Tool Implementations

## Concept: The Three EOP Operations

Your agent performs three complementary operations:

1. **Identify** (bottom-up): Given files, classify each into ECF types
2. **Suggest** (top-down): Given a project description, recommend the ideal structure  
3. **Validate** (gap analysis): Given found artifacts, check if the evidence chain is complete

Together, these create a complete EOP workflow:
```
Identify → Validate → Suggest improvements → Re-validate
```

This mirrors how a human reviewer would assess a paper's reproducibility package — but automated and consistent.

### Implement the Tools
The `identify_artifacts` tool uses the LLM to classify files. The `suggest_structure` tool returns a recommended ECF directory layout. The `validate_chain` tool checks if all 7 artifact types are present. We also redefine the agent here with a refined system prompt.

In [ ]:
def execute_identify_artifacts(args: IdentifyArtifactsArgs) -> str:
    # In production: use LLM to classify each file
    prompt = (
        f"Classify each file into one of these ECF types: "
        f"input_data, experimental_process, output_data, visual_data, "
        f"plotting_process, visual_claim, documentation, unknown.\n"
        f"Files:\n" + "\n".join(f"  - {f}" for f in args.file_descriptions) +
        f"\nContext: {args.context or 'research software project'}\n"
        f"Return a JSON dict mapping each file to its ECF type."
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def execute_suggest_structure(args: SuggestStructureArgs) -> str:
    return (
        f"[suggest_structure] For: {args.project_summary[:80]}\n"
        "  Recommended ECF structure:\n"
        "  work/\n"
        "  input/        <- input_data artifacts\n"
        "  output/       <- output_data artifacts\n"
        "  claim/        <- visual_claim artifacts (figures, tables)\n"
        "  source/       <- experimental_process and plotting_process code\n"
        "  test/         <- reproducibility tests\n"
        "  case/         <- case study configurations\n"
        "  document/     <- documentation\n"
        "  run_all.py    <- pipeline entry point\n"
        "  README.md     <- entry document"
    )

def execute_validate_chain(args: ValidateChainArgs) -> str:
    required = {"input_data","experimental_process","output_data",
                "visual_data","plotting_process","visual_claim","documentation"}
    found = set(args.artifacts_found)
    missing = required - found
    if not missing:
        return f"[validate_chain] COMPLETE. All 7 ECF artifact types present for claim: {args.claim_text[:60]!r}"
    return (
        f"[validate_chain] INCOMPLETE. Missing: {sorted(missing)}\n"
        f"  Claim: {args.claim_text[:60]!r}\n"
        f"  Found: {sorted(found)}\n"
        "  Action: Add the missing artifact types to complete the evidence chain."
    )

def run_tool(name, args):
    if name == "identify_artifacts": return execute_identify_artifacts(args)
    if name == "suggest_structure": return execute_suggest_structure(args)
    if name == "validate_chain": return execute_validate_chain(args)
    return f"[error] Unknown: {name}"

EOP_SYSTEM = ("You are an EOP assistant. Help researchers identify ECF artifacts, suggest directory structures, "
              "and validate evidence chains. Use tools for analysis; answer conceptual questions directly.")

def eop_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": EOP_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "args": None, "content": msg.content}
    call = msg.tool_calls[0]
    raw = json.loads(call.function.arguments)
    validated = SCHEMA_MAP[call.function.name](**raw)
    return {"tool": call.function.name, "args": validated}

## Experiments

### Experiment 1: Classify Repository Files
We give the agent a messy repository with 10 files and ask it to classify each into ECF artifact types. The agent should call the `identify_artifacts` tool and the LLM will map files like `measurements.csv`, `run_experiment.py`, and `fig1_accuracy.png` to their evidence roles.

In [ ]:
# Experiment 1: Identify artifacts in a messy repo
repo_files = [
    "data/raw/measurements.csv",
    "notebooks/analysis_v3_FINAL.ipynb",
    "notebooks/analysis_v2.ipynb",
    "scripts/run_experiment.py",
    "results/model_weights.pkl",
    "results/figure1_raw_data.csv",
    "scripts/make_figures.py",
    "figures/fig1_accuracy.png",
    "figures/fig2_comparison.pdf",
    "README.txt",
]
r = eop_agent(f"Classify these repository files into ECF artifact types: {repo_files}")
print("Tool:", r["tool"])
if r["tool"]:
    result = run_tool(r["tool"], r["args"])
    print("Classification:\n", result)

<details>
<summary>Expected output (click to expand)</summary>

```
Tool: identify_artifacts
Classification:
 {
  "data/raw/measurements.csv": "input_data",
  "notebooks/analysis_v3_FINAL.ipynb": "experimental_process",
  "notebooks/analysis_v2.ipynb": "experimental_process",
  "scripts/run_experiment.py": "experimental_process",
  "results/model_weights.pkl": "output_data",
  "results/figure1_raw_data.csv": "visual_data",
  "scripts/make_figures.py": "plotting_process",
  "figures/fig1_accuracy.png": "visual_claim",
  "figures/fig2_comparison.pdf": "visual_claim",
  "README.txt": "documentation"
}
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment 2: Validate Evidence Chain
Check if a specific claim has complete computational support. We deliberately leave out `visual_data` and `plotting_process` to see if the agent catches the gap. The validate tool should report INCOMPLETE and list exactly which artifact types are missing.

In [ ]:
# Experiment 2: Validate an incomplete chain
r = eop_agent(
    "Check if our evidence chain is complete for the claim "
    "'Figure 2 shows our model achieves 92% accuracy on the benchmark dataset.' "
    "We have: input_data, experimental_process, output_data, visual_claim, documentation."
)
if r["tool"]:
    result = run_tool(r["tool"], r["args"])
    print(result)

<details>
<summary>Expected output (click to expand)</summary>

```
[validate_chain] INCOMPLETE. Missing: ['plotting_process', 'visual_data']
  Claim: 'Figure 2 shows our model achieves 92% accuracy on the benchmark'
  Found: ['documentation', 'experimental_process', 'input_data', 'output_data', 'visual_claim']
  Action: Add the missing artifact types to complete the evidence chain.
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment 3: Suggest Directory Structure
Ask the agent to recommend an ECF-compliant directory structure for a neural network paper. The `suggest_structure` tool will output a standard layout with directories for input data, output data, claims, source code, tests, and documentation.

In [ ]:
# Experiment 3: Suggest directory restructuring
r = eop_agent(
    "Suggest an ECF-compliant directory structure for a neural network paper "
    "where we claim 'Our motif-augmented network outperforms standard ResNet on CIFAR-10.' "
    "Current files: data/, notebooks/, src/, outputs/, paper_figures/, README.md"
)
if r["tool"]:
    result = run_tool(r["tool"], r["args"])
    print(result)
else:
    print("Conversational:", r.get("content"))

<details>
<summary>Expected output (click to expand)</summary>

```
[suggest_structure] For: Our motif-augmented network outperforms standard ResNet on CIFAR-10.
  Recommended ECF structure:
  work/
  input/        <- input_data artifacts
  output/       <- output_data artifacts
  claim/        <- visual_claim artifacts (figures, tables)
  source/       <- experimental_process and plotting_process code
  test/         <- reproducibility tests
  case/         <- case study configurations
  document/     <- documentation
  run_all.py    <- pipeline entry point
  README.md     <- entry document
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **Think about your own research repository.** Which of the 7 ECF artifact types are you currently missing? What would it take to add them?
2. **The validate_chain tool found missing artifacts.** In practice, what is the most commonly missing artifact type? Why?
3. **How would you extend this agent** to handle multi-claim papers where different claims require different evidence chains?

## Summary

You built an EOP agent that:
1. **Identifies** ECF artifacts in repository files using LLM classification
2. **Suggests** ECF-compliant directory structure for any research project
3. **Validates** completeness of evidence chains for scientific claims

**Next**: EOP Lab 2 -- determine required disclosure scope based on claim strength.

---
*Agentic AI Science Playbook -- EOP Domain, Lab EOP1.*